<a href="https://colab.research.google.com/github/k1-r1ll/pytorch/blob/main/linear_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Простая линейная однослойная полносвязная модель

In [4]:
import torch
import torch.nn as nn

# 1. СОЗДАЕМ КЛАСС НАШЕЙ НЕЙРОСЕТИ
# В PyTorch любая модель оформляется как класс, наследуемый от nn.Module.
class LinearRegressionNet(nn.Module):
    def __init__(self, input_dim):
        """
        Конструктор класса. Здесь мы объявляем слои, из которых состоит сеть.
        input_dim — количество входящих признаков (фичей).
        """
        super(LinearRegressionNet, self).__init__()

        # nn.Linear — это аналог Dense-слоя в Keras (линейный / полносвязный слой).
        # Первый аргумент (input_dim): сколько чисел принимает слой на вход.
        # Второй аргумент (1): сколько чисел возвращает на выходе (нам нужно 1 число-прогноз).
        # По умолчанию этот слой уже включает в себя вектор сдвига (bias).
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        """
        Этот метод описывает "прямой проход" (forward pass) данных через сеть.
        Здесь данные x проходят через наш единственный линейный слой.
        """
        # Передаем входные данные x в наш линейный слой и возвращаем результат
        return self.linear(x)

# 2. ИНИЦИАЛИЗИРУЕМ МОДЕЛЬ
# Предположим, у нас, как и в примере выше, 5 входных признаков.
# Создаем экземпляр нашего класса.
model = LinearRegressionNet(input_dim=5)

# 3. ОПРЕДЕЛЯЕМ ФУНКЦИЮ ПОТЕРЬ (КРИТЕРИЙ ОШИБКИ)
# nn.MSELoss() вычисляет среднеквадратичную ошибку между предсказанием сети и правильным ответом.
criterion = nn.MSELoss()

# 4. НАСТРАИВАЕМ ОПТИМИЗАТОР
# torch.optim.Adam — алгоритм оптимизации (вариант градиентного спуска).
# model.parameters() — передаем оптимизатору веса нашей модели, чтобы он знал, что именно нужно обновлять.
# lr=0.01 (Learning Rate) — шаг обучения. Определяет, насколько сильно веса будут меняться за один шаг.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [5]:
# 2. Та же модель, но генерируем для нее данные, инициализируем модель, тренируем и предсказываем

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import r2_score

# ==========================================
# 1. АРХИТЕКТУРА МОДЕЛИ (уже знакомая вам)
# ==========================================
class LinearRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionNet, self).__init__()
        self.linear = nn.Linear(input_dim, 1) # 2 входа (площадь, возраст) -> 1 выход (цена)

    def forward(self, x):
        return self.linear(x)


# ==========================================
# 2. ПОДГОТОВКА ДАННЫХ
# ==========================================
# Нам нужно предсказать цену квартиры. Пусть реальная формула будет:
# Цена = (Площадь * 3) - (Возраст * 0.5) + 10 + небольшой случайный шум

np.random.seed(42) # Фиксируем случайность для воспроизводимости

# Генерируем 100 случайных квартир:
# 1-й столбец: площадь (от 30 до 130 кв.м.)
# 2-й столбец: возраст дома (от 1 до 50 лет)
X_np = np.random.rand(100, 2)
X_np[:, 0] = X_np[:, 0] * 100 + 30
X_np[:, 1] = X_np[:, 1] * 50

# Считаем правильные ответы (цену в тыс. долларов) по нашей формуле с шумом
y_np = (X_np[:, 0] * 3) - (X_np[:, 1] * 0.5) + 10 + np.random.randn(100) * 2
y_np = y_np.reshape(-1, 1) # Превращаем в вектор-столбец

# ⚠️ ВАЖНО: PyTorch работает только со своими Тензорами (тип float32)
X_train = torch.tensor(X_np, dtype=torch.float32)
y_train = torch.tensor(y_np, dtype=torch.float32)


# ==========================================
# 3. ИНИЦИАЛИЗАЦИЯ И НАСТРОЙКА ОБУЧЕНИЯ
# ==========================================
model = LinearRegressionNet(input_dim=2) # У нас 2 признака
criterion = nn.MSELoss()                 # Функция потерь (среднеквадратичная ошибка)
optimizer = optim.Adam(model.parameters(), lr=0.1) # Оптимизатор с шагом обучения 0.1


# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ (TRAINING LOOP)
# ==========================================
epochs = 5000 # Сколько раз модель просмотрит все данные

for epoch in range(epochs):
    # Прямой проход: модель делает предсказание на основе текущих весов
    predictions = model(X_train)

    # Считаем ошибку (насколько модель промахнулась)
    loss = criterion(predictions, y_train)

    # Сбрасываем старые градиенты, чтобы они не копились
    optimizer.zero_grad()

    # Обратный проход: считаем, в какую сторону нужно изменить веса (производные)
    loss.backward()

    # Шаг оптимизатора: обновляем веса модели
    optimizer.step()

    # Выводим лог каждые 500 эпох, чтобы видеть прогресс
    if (epoch + 1) % 500 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}], Ошибка (Loss): {loss.item():.4f}")


# ==========================================
# 5. ИСПОЛЬЗОВАНИЕ МОДЕЛИ (ИНФЕРЕНС / ПРЕДСКАЗАНИЕ)
# ==========================================
# Переводим модель в режим оценки (выключает подсчет градиентов для экономии памяти)
model.eval()

print("\n--- Модель обучена! Делаем предсказание ---")

# У нас появилась новая квартира: Площадь = 85 кв.м., Возраст = 5 лет.
# Обязательно упаковываем данные в тензор такой же размерности, как при обучении (1 строка, 2 колонки)
new_apartment = torch.tensor([[85.0, 5.0]], dtype=torch.float32)

# Отключаем отслеживание градиентов, так как мы только предсказываем, а не учимся
with torch.no_grad():
    predicted_price = model(new_apartment)

# Извлекаем чистое число из тензора с помощью .item()
print(f"Новый объект: Площадь 85 кв.м, Возраст 5 лет.")
print(f"Предсказанная стоимость: ${predicted_price.item():.2f} тыс.")

# Наша истинная формула без шума дала бы: (85 * 3) - (5 * 0.5) + 10 = 255 - 2.5 + 10 = 262.5


Эпоха [500/5000], Ошибка (Loss): 7.8703
Эпоха [1000/5000], Ошибка (Loss): 6.1744
Эпоха [1500/5000], Ошибка (Loss): 4.8939
Эпоха [2000/5000], Ошибка (Loss): 4.2267
Эпоха [2500/5000], Ошибка (Loss): 3.9884
Эпоха [3000/5000], Ошибка (Loss): 3.9335
Эпоха [3500/5000], Ошибка (Loss): 3.9261
Эпоха [4000/5000], Ошибка (Loss): 3.9255
Эпоха [4500/5000], Ошибка (Loss): 3.9255
Эпоха [5000/5000], Ошибка (Loss): 3.9255

--- Модель обучена! Делаем предсказание ---
Новый объект: Площадь 85 кв.м, Возраст 5 лет.
Предсказанная стоимость: $262.49 тыс.


In [7]:
# 3. То же самое, но добавим разбивку выбоки на тренировку и тест

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
# Добавляем инструменты для честного разбиения и масштабирования
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ==========================================
# 1. АРХИТЕКТУРА МОДЕЛИ
# ==========================================
class LinearRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionNet, self).__init__()
        self.linear = nn.Linear(input_dim, 1) # 2 входа -> 1 выход

    def forward(self, x):
        return self.linear(x)


# ==========================================
# 2. ПОДГОТОВКА И ЧЕСТНОЕ РАЗБИЕНИЕ ДАННЫХ
# ==========================================
np.random.seed(42)

# Генерируем исходный датасет (100 квартир)
X_np = np.random.rand(100, 2)
X_np[:, 0] = X_np[:, 0] * 100 + 30  # Площадь (30-130)
X_np[:, 1] = X_np[:, 1] * 50        # Возраст (0-50)

y_np = (X_np[:, 0] * 3) - (X_np[:, 1] * 0.5) + 10 + np.random.randn(100) * 2
y_np = y_np.reshape(-1, 1)

# --- ШАГ 1: Разделяем данные на Train и Test (80% на 20%) ---
# Делаем это ДО масштабирования, чтобы избежать утечки данных!
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

# --- ШАГ 2: Масштабирование признаков (Scaling) ---
# Площадь (до 130) и Возраст (до 50) имеют разный шаг. Нейросетям тяжело учиться на разномасштабных данных.
scaler = StandardScaler()

# Находим среднее и отклонение ТОЛЬКО на обучающей выборке и трансформируем её (то есть вычитаем среднее и делим на СО для этой подвыборки, X_train_scaled)
X_train_scaled = scaler.fit_transform(X_train_np)
# Тестовую выборку только трансформируем (модель не должна знать среднее тестовой выборки!; тут применяем среднее и СО с подвыборки выше, то есть не пересчи. их с учетом X_test_scaled)
X_test_scaled = scaler.transform(X_test_np)

# --- ШАГ 3: Конвертация в тензоры PyTorch ---
X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32)

X_test = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32)


# ==========================================
# 3. ИНИЦИАЛИЗАЦИЯ И НАСТРОЙКА ОБУЧЕНИЯ
# ==========================================
model = LinearRegressionNet(input_dim=2)
criterion = nn.MSELoss()
# Так как данные отмасштабированы, шаг обучения (lr) можно сделать чуть меньше для плавной сходимости
optimizer = optim.Adam(model.parameters(), lr=0.05)


# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ С ВАЛИДАЦИЕЙ
# ==========================================
epochs = 2000 # Благодаря скейлингу модель обучится гораздо быстрее, 2000 эпох хватит с запасом

for epoch in range(epochs):
    # --- ЭТАП ОБУЧЕНИЯ (TRAIN) ---
    model.train() # Включаем режим обучения

    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # --- ЭТАП ЧЕСТНОЙ ПРОВЕРКИ (TEST/VALIDATION) ---
    # Каждые 200 эпох проверяем, как модель справляется с данными, которых она не видит
    if (epoch + 1) % 200 == 0:
        model.eval() # Выключаем режим обучения (дропауты, батч-норм, если бы они были)

        with torch.no_grad(): # Отключаем расчет градиентов для теста (экономит память и время)
            test_predictions = model(X_test)
            test_loss = criterion(test_predictions, y_test)

        print(f"Эпоха [{epoch+1}/{epochs}] | Ошибка на обучении (Train Loss): {loss.item():.4f} | Ошибка на тесте (Test Loss): {test_loss.item():.4f}")


# ==========================================
# 5. ИСПОЛЬЗОВАНИЕ МОДЕЛИ (ИНФЕРЕНС)
# ==========================================
model.eval()

print("\n--- Модель честно обучена и проверена! ---")

# Новая квартира: Площадь = 85 кв.м., Возраст = 5 лет.
new_apartment_raw = np.array([[85.0, 5.0]])

# ⚠️ ВАЖНО: Новые данные перед подачей в модель нужно нормализовать ТЕМ ЖЕ скалером!
new_apartment_scaled = scaler.transform(new_apartment_raw)
new_apartment_tensor = torch.tensor(new_apartment_scaled, dtype=torch.float32)

with torch.no_grad():
    predicted_price = model(new_apartment_tensor)

print(f"Новый объект (оригинал): Площадь 85 кв.м, Возраст 5 лет.")
print(f"Предсказанная стоимость: ${predicted_price.item():.2f} тыс.")
print(f"Истинная стоимость по формуле (без шума): $262.50 тыс.")


Эпоха [200/2000] | Ошибка на обучении (Train Loss): 57134.9141 | Ошибка на тесте (Test Loss): 54066.3672
Эпоха [400/2000] | Ошибка на обучении (Train Loss): 51403.7773 | Ошибка на тесте (Test Loss): 48913.3359
Эпоха [600/2000] | Ошибка на обучении (Train Loss): 46183.1680 | Ошибка на тесте (Test Loss): 44205.7852
Эпоха [800/2000] | Ошибка на обучении (Train Loss): 41431.3516 | Ошибка на тесте (Test Loss): 39886.1016
Эпоха [1000/2000] | Ошибка на обучении (Train Loss): 37109.7305 | Ошибка на тесте (Test Loss): 35924.4062
Эпоха [1200/2000] | Ошибка на обучении (Train Loss): 33182.4453 | Ошибка на тесте (Test Loss): 32293.0840
Эпоха [1400/2000] | Ошибка на обучении (Train Loss): 29615.9570 | Ошибка на тесте (Test Loss): 28966.3906
Эпоха [1600/2000] | Ошибка на обучении (Train Loss): 26378.9746 | Ошибка на тесте (Test Loss): 25920.3594
Эпоха [1800/2000] | Ошибка на обучении (Train Loss): 23442.2246 | Ошибка на тесте (Test Loss): 23132.6484
Эпоха [2000/2000] | Ошибка на обучении (Train Loss

In [9]:
# 4. Тут добавил проверку для ранней остановки

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [11]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=0)

In [12]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target

In [13]:
class LinearRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegressionNet, self).__init__()
        self.linear = nn.Linear(input_dim, 1) # 2 входа -> 1 выход

    def forward(self, x):
        return self.linear(x)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [15]:
X_train = X_train.to(device)
y_train = y_train.to(device)

X_test = X_test.to(device)
y_test = y_test.to(device)

In [16]:
model = LinearRegressionNet(input_dim=8).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

In [17]:
epochs = 2000

for epoch in range(epochs):
    model.train()

    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        model.eval()

        with torch.no_grad():
            test_predictions = model(X_test)
            test_loss = criterion(test_predictions, y_test)

        print(f"Эпоха [{epoch+1}/{epochs}] | Ошибка на обучении (Train Loss): {loss.item():.4f} | Ошибка на тесте (Test Loss): {test_loss.item():.4f}")

Эпоха [200/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [400/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [600/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [800/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [1000/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [1200/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [1400/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [1600/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [1800/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559
Эпоха [2000/2000] | Ошибка на обучении (Train Loss): 0.5179 | Ошибка на тесте (Test Loss): 0.5559


In [18]:
# --- Настройки для ранней остановки (Early Stopping) ---
patience = 3                 # Сколько проверок мы готовы ждать без улучшений
checks_without_progress = 0  # Счетчик "плохих" проверок
best_test_loss = float('inf')# Изначально лучшая ошибка равна бесконечности
min_delta = 0.001            # Минимальное улучшение, которое мы считаем значимым

epochs = 2000

for epoch in range(epochs):
    model.train()
    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Каждые 200 эпох делаем честную проверку
    if (epoch + 1) % 200 == 0:
        model.eval()
        with torch.no_grad():
            test_predictions = model(X_test)
            test_loss = criterion(test_predictions, y_test)

        current_test_loss = test_loss.item()
        print(f"Эпоха [{epoch+1}/{epochs}] | Train Loss: {loss.item():.4f} | Test Loss: {current_test_loss:.4f}")

        # --- ЛОГИКА РАННЕЙ ОСТАНОВКИ ---
        # Проверяем, стала ли ошибка значимо меньше, чем наш лучший результат
        if current_test_loss < best_test_loss - min_delta:
            best_test_loss = current_test_loss  # Запоминаем новый лучший результат
            checks_without_progress = 0          # Сбрасываем счетчик ожидания
        else:
            checks_without_progress += 1         # Улучшения нет, добавляем единицу к ожиданию

        # Если терпение кончилось — останавливаем обучение
        if checks_without_progress >= patience:
            print(f"\n🛑 Обучение остановлено на эпохе {epoch+1}!")
            print(f"Ошибка перестала падать. Лучший Test Loss: {best_test_loss:.4f}")
            break


Эпоха [200/2000] | Train Loss: 0.5179 | Test Loss: 0.5559
Эпоха [400/2000] | Train Loss: 0.5179 | Test Loss: 0.5559
Эпоха [600/2000] | Train Loss: 0.5179 | Test Loss: 0.5559
Эпоха [800/2000] | Train Loss: 0.5179 | Test Loss: 0.5559

🛑 Обучение остановлено на эпохе 800!
Ошибка перестала падать. Лучший Test Loss: 0.5559


In [19]:
# 5. Тут немного усложнил модель, для нелинейности добавил функцию активации

In [20]:
class DeepRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(DeepRegressionNet, self).__init__()
        # Первый слой: с 8 признаков расширяем до 64 скрытых нейронов
        self.fc1 = nn.Linear(input_dim, 64)
        # Функция активации, которая "сгибает" пространство и находит нелинейные связи
        self.relu = nn.ReLU()
        # Выходной слой: с 64 скрытых нейронов собираем в 1 ответ
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out


In [21]:
model = DeepRegressionNet(input_dim=X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

In [22]:
# --- Настройки для ранней остановки (Early Stopping) ---
patience = 3                 # Сколько проверок мы готовы ждать без улучшений
checks_without_progress = 0  # Счетчик "плохих" проверок
best_test_loss = float('inf')# Изначально лучшая ошибка равна бесконечности
min_delta = 0.001            # Минимальное улучшение, которое мы считаем значимым

epochs = 2000

for epoch in range(epochs):
    model.train()
    predictions = model(X_train)
    loss = criterion(predictions, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Каждые 200 эпох делаем честную проверку
    if (epoch + 1) % 200 == 0:
        model.eval()
        with torch.no_grad():
            test_predictions = model(X_test)
            test_loss = criterion(test_predictions, y_test)

        current_test_loss = test_loss.item()
        print(f"Эпоха [{epoch+1}/{epochs}] | Train Loss: {loss.item():.4f} | Test Loss: {current_test_loss:.4f}")

        # --- ЛОГИКА РАННЕЙ ОСТАНОВКИ ---
        # Проверяем, стала ли ошибка значимо меньше, чем наш лучший результат
        if current_test_loss < best_test_loss - min_delta:
            best_test_loss = current_test_loss  # Запоминаем новый лучший результат
            checks_without_progress = 0          # Сбрасываем счетчик ожидания
        else:
            checks_without_progress += 1         # Улучшения нет, добавляем единицу к ожиданию

        # Если терпение кончилось — останавливаем обучение
        if checks_without_progress >= patience:
            print(f"\n🛑 Обучение остановлено на эпохе {epoch+1}!")
            print(f"Ошибка перестала падать. Лучший Test Loss: {best_test_loss:.4f}")
            break


Эпоха [200/2000] | Train Loss: 0.2939 | Test Loss: 0.3171
Эпоха [400/2000] | Train Loss: 0.2646 | Test Loss: 0.2897
Эпоха [600/2000] | Train Loss: 0.2522 | Test Loss: 0.2808
Эпоха [800/2000] | Train Loss: 0.2592 | Test Loss: 0.2910
Эпоха [1000/2000] | Train Loss: 0.2447 | Test Loss: 0.2867
Эпоха [1200/2000] | Train Loss: 0.2952 | Test Loss: 0.3323

🛑 Обучение остановлено на эпохе 1200!
Ошибка перестала падать. Лучший Test Loss: 0.2808


In [25]:
# 6. Тут еще больше усложнил модель; расширил до трех слоев; тренировку собрал в одну функцию; лобавил вывод коэффициента детерминации

In [26]:
class DeeperRegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(DeeperRegressionNet, self).__init__()
        # Входной слой: с 8 признаков до 128 нейронов
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()

        # Второй слой: со 128 до 64 нейронов
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        # Третий слой: с 64 до 32 нейронов
        self.fc3 = nn.Linear(64, 32)
        self.relu3 = nn.ReLU()

        # Выходной слой: с 32 нейронов собираем 1 ответ
        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):
        out = self.relu1(self.fc1(x))
        out = self.relu2(self.fc2(out))
        out = self.relu3(self.fc3(out))
        out = self.fc4(out)
        return out


In [27]:
def train(model, criterion, optimizer, epochs,
          patience = 3, checks_without_progress = 0,
          min_delta = 0.001):

  best_test_loss = float('inf')

  for epoch in range(epochs):
      model.train()
      predictions = model(X_train)
      loss = criterion(predictions, y_train)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      if (epoch + 1) % 200 == 0:
          model.eval()
          with torch.no_grad():
              test_predictions = model(X_test)
              test_loss = criterion(test_predictions, y_test)

          current_test_loss = test_loss.item()

          # --- СЧИТАЕМ КОЭФФИЦИЕНТ ДЕТЕРМИНАЦИИ R² ---
          # Переводим тензоры на CPU и конвертируем в NumPy для sklearn
          y_true_np = y_test.cpu().numpy()
          y_pred_np = test_predictions.cpu().numpy()

          r2 = r2_score(y_true_np, y_pred_np)

          # Выводим лог с добавлением точности в процентах
          print(f"Эпоха [{epoch+1}/{epochs}] | "
                f"Train Loss: {loss.item():.4f} | "
                f"Test Loss: {current_test_loss:.4f} | "
                f"Точность (R²): {r2 * 100:.2f}%")

          if current_test_loss < best_test_loss - min_delta:
              best_test_loss = current_test_loss
              checks_without_progress = 0
          else:
              checks_without_progress += 1

          if checks_without_progress >= patience:
              print(f"\n🛑 Обучение остановлено на эпохе {epoch+1}!")
              print(f"Ошибка перестала падать. Лучший Test Loss: {best_test_loss:.4f}")
              break


In [28]:
epochs = 2000
model = DeeperRegressionNet(input_dim=X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = train(model, criterion, optimizer, epochs)

Эпоха [200/2000] | Train Loss: 0.3823 | Test Loss: 0.3926 | Точность (R²): 70.04%
Эпоха [400/2000] | Train Loss: 0.2915 | Test Loss: 0.3090 | Точность (R²): 76.42%
Эпоха [600/2000] | Train Loss: 0.2565 | Test Loss: 0.2857 | Точность (R²): 78.20%
Эпоха [800/2000] | Train Loss: 0.2424 | Test Loss: 0.2808 | Точность (R²): 78.57%
Эпоха [1000/2000] | Train Loss: 0.2326 | Test Loss: 0.2788 | Точность (R²): 78.73%
Эпоха [1200/2000] | Train Loss: 0.2256 | Test Loss: 0.2787 | Точность (R²): 78.73%
Эпоха [1400/2000] | Train Loss: 0.2197 | Test Loss: 0.2797 | Точность (R²): 78.65%
Эпоха [1600/2000] | Train Loss: 0.2153 | Test Loss: 0.2869 | Точность (R²): 78.11%

🛑 Обучение остановлено на эпохе 1600!
Ошибка перестала падать. Лучший Test Loss: 0.2788
